In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 871 unique observed indicators.


In [3]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,source,...,Block,ip,lastObserved,text,sha256,md5,sha1,size,address,indicator
26,5629499555060733,2025-06-16T17:42:56Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-22T14:58:03Z,3.0,76.0,NaN,...,NaN,192.42.116.198,2025-09-20T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.198
34,4353203,2023-06-09T15:50:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-22T14:57:19Z,5.0,76.0,NaN,...,NaN,51.83.189.185,2025-09-20T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,51.83.189.185
38,4720930,2024-06-17T18:10:46Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-22T14:56:58Z,4.0,54.0,NaN,...,NaN,185.241.208.204,2025-09-22T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,185.241.208.204
52,234532,2017-01-03T15:27:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-22T14:56:48Z,3.0,64.0,Department of Homeland Security \r\nNCCIC-- US...,...,NaN,80.67.172.162,2025-09-22T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,80.67.172.162
58,5629499542033704,2025-05-28T17:45:50Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-22T14:54:52Z,3.0,64.0,NaN,...,NaN,45.91.250.107,2025-09-21T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,45.91.250.107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9715,6755399448005403,2025-05-19T11:53:21Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-18T12:18:07Z,3.0,63.0,NaN,...,NaN,45.80.158.167,2025-09-22T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,45.80.158.167
9780,4397426,2023-08-28T13:07:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-18T11:53:10Z,3.0,21.0,NaN,...,NaN,72.21.210.29,2025-09-22T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,72.21.210.29
9785,4564864,2024-04-15T15:13:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-18T11:53:05Z,5.0,78.0,NaN,...,NaN,66.235.168.222,2025-09-20T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,66.235.168.222
9801,6755399467420957,2025-08-13T14:52:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-18T11:48:27Z,3.0,65.0,NaN,...,NaN,NaN,2025-09-22T00:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,proudlyhint.com


In [4]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_29956\564055639.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250922.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250921.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250920.csv']
Loaded data from 3 files.


In [6]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ── Time cutoffs ──────────────────────────────────────────────────────────────
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=365)

# ── Collect filtered rows fast, then concat once ─────────────────────────────
all_filtered = []

print(f"observed_src shape: {observed_src.shape}")

# ── Loop through each row in observed_src ─────────────────────────────────────
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    if not isinstance(tags_data, list):
        continue

    # Normalize the tags for the current row
    tags_df = pd.json_normalize(tags_data)

    # Ensure string and apply VA rename before filtering
    tags_df['name'] = (
        tags_df['name']
        .astype(str)
        .str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)
    )

    # Skip if all associated groups are Adversary
    if 'associatedGroups.data' in observed_src.columns:
        ag_data = row.get('associatedGroups.data')
        if isinstance(ag_data, list) and len(ag_data) > 0:
            groups_df = pd.json_normalize(ag_data)
            if 'type' in groups_df.columns and set(groups_df['type']) == {'Adversary'}:
                continue

    # ── FIXED: robust group_id extraction without later overwrite ─────────────
    def pick_group_id(ag):
        if not isinstance(ag, list):
            return None
        for g in ag:
            if isinstance(g, dict):
                gid = g.get('id') or (g.get('group') or {}).get('id')
                if gid is None:
                    continue
                s = str(gid)
                if s.isdigit() and len(s) == 16:
                    return s
        return None

    group_id = pick_group_id(row.get('associatedGroups.data'))

    # All tag names (for all_tags)
    all_tags_list = tags_df['name'].tolist()

    # Filter for API tags
    api_tags = tags_df[tags_df['name'].str.contains('API', case=False, na=False)].copy()
    if api_tags.empty:
        continue

    # Add metadata columns (including rating, confidence, id)
    # NOTE: 'group_id' REMOVED from meta_cols to prevent overwrite
    meta_cols = [
        'summary', 'observations', 'description', 'type',
        'dateAdded', 'lastModified', 'lastObserved', 'webLink',
        'rating', 'confidence', 'id'
    ]
    for col in meta_cols:
        api_tags[col] = row.get(col)

    # Set group_id AFTER metadata copy so it can't be overwritten
    api_tags['group_id'] = group_id

    # Attach all tags list
    api_tags['all_tags'] = [all_tags_list] * len(api_tags)

    # Accumulate
    all_filtered.append(api_tags)

# ── Create filtered_tags DataFrame ────────────────────────────────────────────
filtered_tags = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()
print(f"filtered_tags shape: {filtered_tags.shape}")

# Ensure datetimes
if not filtered_tags.empty:
    filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce', utc=True)
    filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce', utc=True)

# ── Validate observed_data_df has needed columns ──────────────────────────────
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [c for c in required_columns if c not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

print(f"observed_data_df shape: {observed_data_df.shape}")

# ── Clean name -> match OpDiv (remove trailing ' Splunk API') ─────────────────
if not filtered_tags.empty:
    filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

    # Map observed_date by (indicator=summary, OpDiv=cleaned_name)
    filtered_tags['observed_date'] = pd.NaT
    # Ensure observed_data_df obs_date is datetime (naive)
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')

    for idx, r in filtered_tags.iterrows():
        summary = r.get('summary')
        cleaned_name = r.get('cleaned_name')
        if pd.isna(summary) or pd.isna(cleaned_name):
            continue
        match = observed_data_df[
            (observed_data_df['indicator'] == summary) &
            (observed_data_df['OpDiv'] == cleaned_name)
        ]
        if not match.empty:
            filtered_tags.at[idx, 'observed_date'] = match['obs_date'].iloc[0]

    filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

    # Drop helper
    filtered_tags.drop(columns=['cleaned_name'], inplace=True, errors='ignore')
    print(f"filtered_tags shape after observed_date mapping: {filtered_tags.shape}")

# ── Recent filters ────────────────────────────────────────────────────────────
# Last 24h by lastObserved (aware)
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)].copy()
print(f"recent_tags shape after lastObserved filter: {recent_tags.shape}")

# Make cutoff naive to compare against observed_date (naive)
cutoff_naive = cutoff.tz_convert(None)

# Last 2 days by observed_date (naive)
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()
print(f"recent_tags shape after observed_date filter: {recent_tags.shape}")

# ── Partner extraction and grouping ───────────────────────────────────────────
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)
print(f"recent_tags shape after partner column: {recent_tags.shape}")

partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)
print(f"partner_groups shape: {partner_groups.shape}")

recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')
print(f"recent_tags shape after merge: {recent_tags.shape}")

# ── Additional filters ────────────────────────────────────────────────────────
recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
print(f"recent_tags shape after partner_count filter: {recent_tags.shape}")

recent_tags = recent_tags[recent_tags['observations'] < 15000]
print(f"recent_tags shape after observations filter: {recent_tags.shape}")

recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]
print(f"recent_tags shape after dateAdded filter: {recent_tags.shape}")

# Deduplicate by summary
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')
print(f"recent_tags shape after deduplication: {recent_tags.shape}")

# Drop unneeded columns if present
cols_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name'
]
recent_tags = recent_tags.drop(columns=[c for c in cols_to_drop if c in recent_tags.columns], errors='ignore')
print(f"recent_tags shape after dropping columns: {recent_tags.shape}")

# Remove rows where all_tags contains unwanted markers
if 'all_tags' in recent_tags.columns:
    recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'I&W' in x)]
    print(f"recent_tags shape after I&W filter: {recent_tags.shape}")
    recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: isinstance(x, list) and 'htoc_wl' in x)]
    print(f"recent_tags shape after htoc_wl filter: {recent_tags.shape}")

# Preview
print(f"recent_tags final shape: {recent_tags.shape}")
recent_tags


observed_src shape: (871, 32)
filtered_tags shape: (6697, 20)
observed_data_df shape: (13801, 7)
filtered_tags shape after observed_date mapping: (6697, 21)
recent_tags shape after lastObserved filter: (5992, 21)
recent_tags shape after observed_date filter: (4420, 21)
recent_tags shape after partner column: (4420, 22)
partner_groups shape: (779, 3)
recent_tags shape after merge: (4420, 24)
recent_tags shape after partner_count filter: (4275, 24)
recent_tags shape after observations filter: (149, 24)
recent_tags shape after dateAdded filter: (144, 24)
recent_tags shape after deduplication: (54, 24)
recent_tags shape after dropping columns: (54, 17)
recent_tags shape after I&W filter: (51, 17)
recent_tags shape after htoc_wl filter: (51, 17)
recent_tags final shape: (51, 17)


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,group_id,all_tags,observed_date,partner_count,partners
4,5629499542033704,TASK0881388 / RITM0584642,2025-09-22T11:16:48Z,45.91.250.107,6686,Address,2025-05-28 17:45:50+00:00,2025-09-22T14:54:52Z,2025-09-21 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,None,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-09-21,2,"HRSA, VA"
121,5629499546480626,TASK0882701 / RITM0585661,2025-09-22T07:36:30Z,138.68.174.105,8980,Address,2025-06-02 18:33:40+00:00,2025-09-22T14:21:45Z,2025-09-21 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,64.0,None,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-09-21,7,"CMS, DHA, HHS, HRSA, IHS, OS, VA"
128,6755399468016505,INC9203759,2025-09-22T13:14:50Z,193.46.255.7,3291,Address,2025-08-20 18:25:15+00:00,2025-09-22T14:14:59Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,76.0,None,"[OS Splunk API, VA Splunk API, Observed, malic...",2025-09-22,2,"OS, VA"
132,6755399468016503,INC9203759,2025-09-22T13:14:50Z,193.46.255.33,3277,Address,2025-08-20 18:25:14+00:00,2025-09-22T14:14:59Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,76.0,None,"[OS Splunk API, VA Splunk API, Observed, malic...",2025-09-21,2,"OS, VA"
134,6755399468016502,INC9203759,2025-09-22T13:14:50Z,80.94.93.233,3210,Address,2025-08-20 18:25:13+00:00,2025-09-22T14:14:59Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,76.0,None,"[OS Splunk API, VA Splunk API, Observed, malic...",2025-09-21,2,"OS, VA"
136,5629499564014151,INC9203759,2025-09-22T13:14:50Z,193.46.255.217,13614,Address,2025-08-20 18:25:15+00:00,2025-09-22T14:14:59Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,76.0,None,"[OS Splunk API, CMS Splunk API, VA Splunk API,...",2025-09-22,2,"OS, VA"
159,6755399468016499,INC9203759,2025-09-22T13:14:50Z,193.46.255.99,14535,Address,2025-08-20 18:25:11+00:00,2025-09-22T14:13:58Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,76.0,None,"[OS Splunk API, CMS Splunk API, VA Splunk API,...",2025-09-22,2,"OS, VA"
161,6755399460008019,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-22T11:16:48Z,134.199.159.23,5822,Address,2025-07-02 12:05:33+00:00,2025-09-22T13:59:42Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-22,2,"HRSA, VA"
173,6755399460008452,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-22T13:14:50Z,38.97.116.243,4403,Address,2025-07-02 12:05:37+00:00,2025-09-22T13:49:29Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-22,3,"CMS, OS, VA"
198,6755399448005402,RITM0581774/TASK0877781,2025-09-22T13:14:50Z,104.244.79.50,4954,Address,2025-05-19 11:53:21+00:00,2025-09-22T13:38:43Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,63.0,None,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-09-21,2,"HHS, OS"


In [7]:
import pandas as pd
import urllib.parse

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)


No attributes found for indicator: 185.220.101.30


In [8]:
filtered_recent_tags

,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,rating,confidence,group_id,all_tags,observed_date,partner_count,partners
0,6755399460008019,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-22T11:16:48Z,134.199.159.23,5822,Address,2025-07-02 12:05:33+00:00,2025-09-22T13:59:42Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-22,2,"HRSA, VA"
1,6755399460008452,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-22T13:14:50Z,38.97.116.243,4403,Address,2025-07-02 12:05:37+00:00,2025-09-22T13:49:29Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,1.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-22,3,"CMS, OS, VA"
2,6755399460008358,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-22T02:13:23Z,170.80.76.38,3,Address,2025-07-02 12:05:36+00:00,2025-09-22T11:16:45Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-21,2,"HHS, IHS"
3,5265088,CISA conducted a hunt on IoC's obtained from o...,2025-09-22T13:14:50Z,138.197.101.95,6947,Address,2025-01-23 19:51:13+00:00,2025-09-22T10:55:15Z,2025-09-21 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,53.0,None,"[DHA Splunk API, CVE-2025-283, CVE-2025-282, O...",2025-09-21,4,"CMS, IHS, NIH, OS"
4,5265076,CISA conducted a hunt on IoC's obtained from o...,2025-09-22T03:59:05Z,104.131.6.219,10164,Address,2025-01-23 19:51:12+00:00,2025-09-22T10:54:35Z,2025-09-21 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,3.0,53.0,None,"[DHA Splunk API, CVE-2025-283, CVE-2025-282, O...",2025-09-21,3,"HRSA, IHS, NIH"
5,6755399460007923,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-22T07:36:30Z,74.91.112.216,4,Address,2025-07-02 12:05:33+00:00,2025-09-22T04:07:26Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-22,2,"DHA, HRSA"
6,5629499537014343,CISA observed scripted download and execution ...,2025-09-22T07:36:30Z,104.21.20.66,505,Address,2025-04-21 11:07:48+00:00,2025-09-21T22:20:18Z,2025-09-21 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,54.0,None,"[DHA Splunk API, malicious code, VA OIS CSOC C...",2025-09-21,2,"DHA, HRSA"
7,6755399460007597,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-21T07:36:36Z,38.123.220.202,15,Address,2025-07-02 12:05:30+00:00,2025-09-21T18:42:26Z,2025-09-21 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-21,2,"CMS, VA"
8,6755399460007743,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-21T07:36:36Z,143.110.190.60,7145,Address,2025-07-02 12:05:31+00:00,2025-09-21T18:42:05Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-21,2,"CMS, VA"
9,6755399460007564,"Details\nIn late June 2025, Iran-aligned hackt...",2025-09-22T11:16:48Z,206.189.57.182,4788,Address,2025-07-02 12:05:30+00:00,2025-09-20T18:22:58Z,2025-09-22 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,4.0,62.0,5629499544001057,"[INDICATOR NOTICE 25178.1, Mr Hamza Group, T-S...",2025-09-22,2,"HRSA, VA"


In [9]:
import pandas as pd
import requests
import time
import ipaddress
from datetime import datetime
import concurrent.futures

# API Keys
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"

# Headers for API requests
VT_HEADERS = {"x-apikey": VT_API_KEY}
OTX_HEADERS = {"X-OTX-API-KEY": OTX_API_KEY}

# API URLs
VT_URL_IP = "https://www.virustotal.com/api/v3/ip_addresses/{}"
VT_URL_DOMAIN = "https://www.virustotal.com/api/v3/domains/{}"
OTX_URL_IP = "https://otx.alienvault.io/api/v1/indicators/IPv4/{}/general"
OTX_URL_DOMAIN = "https://otx.alienvault.io/api/v1/indicators/domain/{}/general"
OTX_URL_HOSTNAME = "https://otx.alienvault.io/api/v1/indicators/hostname/{}"

def is_ip(value):
    """ Check if the given value is a valid IP address. """
    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False

def determine_query_type(query):
    """ Determine if the query is an IP, domain, or hostname. """
    if is_ip(query):
        return "ip"
    elif "." in query:
        return "hostname"
    else:
        return "domain"

def fetch_virustotal_data(query):
    """Fetch data from VirusTotal API for IP or Domain using ThreatConnect enrich endpoint."""
    import urllib.parse

    # indicator_values should be a list of indicator values to enrich
    # For compatibility, if query is a single value, wrap in list
    indicator_values = [query] if isinstance(query, str) else query
    enriched_results = []

    for value in indicator_values:
        try:
            # Use the indicator *value*, not the ID
            encoded_value = urllib.parse.quote(value)
            enrich_url = f'/v3/indicators/{encoded_value}/enrich?type=Shodan&type=VirusTotalV3'
            ro.set_http_method('POST')
            ro.set_request_uri(enrich_url)
            ro.set_body({})
            enrich_response = tc.api_request(ro)

            if enrich_response.status_code == 200:
                enrich_data = enrich_response.json()
                enrich_data['summary'] = value  # Save the value as key
                enriched_results.append(enrich_data)
        except Exception as e:
            continue
    # Unnest the 'data' field in enriched_results if it's a list of dicts
    if enriched_results:
        # If enriched_results is a list of dicts with 'data' key, flatten each
        flattened = []
        for item in enriched_results:
            if isinstance(item, dict) and 'data' in item:
                flat = item.copy()
                flat.update(pd.json_normalize(item['data']).iloc[0].to_dict() if isinstance(item['data'], dict) else {})
                del flat['data']
                flattened.append(flat)
            else:
                flattened.append(item)
        enriched_results = flattened

    # Return first result if only one value, else list
    if len(enriched_results) == 1:
        return enriched_results[0]
    
    return enriched_results

def fetch_otx_data(query):
    """Fetch data from OTX API for IP, Domain, or Hostname."""
    query_type = determine_query_type(query)

    if query_type == "ip":
        url = OTX_URL_IP.format(query)
    elif query_type == "hostname":
        url = OTX_URL_HOSTNAME.format(query)
    else:
        url = OTX_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=OTX_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json()

        return {
            "search_term": query,
            "base_indicator": data.get('base_indicator', {}),
            "reputation": data.get('reputation'),
            "geo": data.get('geo', {}),
            "whois": data.get('whois', {}),
            "open_ports": data.get('open_ports', []),
            "link": f"https://otx.alienvault.com/indicator/{query_type}/{query}"
        }

    except Exception as e:
        print(f"OTX Error for {query}: {e}")
        return None

def unnest_base_indicator(df):
    """ Unnest the 'base_indicator' column and create separate columns for its keys. """
    if 'base_indicator' not in df.columns:
        return df

    base_df = pd.json_normalize(df['base_indicator'])
    base_df.columns = [f"base_{col}" for col in base_df.columns]

    df = df.drop(columns=['base_indicator']).reset_index(drop=True)
    df = pd.concat([df, base_df], axis=1)
    return df

def process_indicator(indicator, observed_by, partner_count):
    """Fetch data for a single indicator."""
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        vt_future = executor.submit(fetch_virustotal_data, indicator)
        otx_future = executor.submit(fetch_otx_data, indicator)

        vt_data = vt_future.result()
        otx_data = otx_future.result()

    if vt_data:
        vt_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    if otx_data:
        otx_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    return vt_data, otx_data

def main(recent_tags):
    """ Main function to process indicators. """
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing.")
        return pd.DataFrame(), pd.DataFrame()

    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Processing {len(search_terms)} unique search terms.")

    vt_results = []
    otx_results = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for indicator in search_terms:
            partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
            partner_count = recent_tags.loc[recent_tags['summary'] == indicator, 'partner_count'].values
            observed_by = partners[0] if len(partners) > 0 else "N/A"
            partner_count = partner_count[0] if len(partner_count) > 0 else "N/A"

            futures.append(executor.submit(process_indicator, indicator, observed_by, partner_count))

        for future in concurrent.futures.as_completed(futures):
            vt_data, otx_data = future.result()
            if vt_data:
                vt_results.append(vt_data)
            if otx_data:
                otx_results.append(otx_data)

    vt_df = pd.DataFrame(vt_results)
    otx_df = pd.DataFrame(otx_results)

    
    otx_df = unnest_base_indicator(otx_df)

    return vt_df, otx_df

if __name__ == "__main__":
    vt_df, otx_df = main(filtered_recent_tags)
    print("Script completed successfully.")


Processing 16 unique search terms.


C:\Users\jaskew\AppData\Local\Temp\ipykernel_29956\2999662670.py:129: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


OTX Error for 185.220.101.30: 504 Server Error: Gateway Time-out for url: https://otx.alienvault.io/api/v1/indicators/IPv4/185.220.101.30/general
Script completed successfully.


In [15]:
vt_df

,status,summary,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,...,privateFlag,active,activeLocked,ip,legacyLink,enrichment.data,timestamp,observed_by,partner_count,source
0,Success,134.199.159.23,6755399460008019,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-21T18:42:11Z,4.0,...,False,True,False,134.199.159.23,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['cloud'], 'cloudP...",2025-09-22 11:38:06,"HRSA, VA",2,NaN
1,Success,222.59.173.105,6755399460007583,2025-07-02T12:05:30Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-20T18:22:28Z,4.0,...,False,True,False,222.59.173.105,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'country': 'China', 'city'...",2025-09-22 11:38:08,"HHS, NIH",2,NaN
2,Success,74.91.112.216,6755399460007923,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-22T04:07:26Z,4.0,...,False,True,False,74.91.112.216,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'hostNames': ['v-74-91-112...",2025-09-22 11:38:06,"DHA, HRSA",2,NaN
3,Success,104.21.20.66,5629499537014343,2025-04-21T11:07:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-21T22:20:18Z,4.0,...,False,True,False,104.21.20.66,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['cdn'], 'country'...",2025-09-22 11:38:06,"DHA, HRSA",2,CISA INAR
4,Success,143.110.190.60,6755399460007743,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-21T18:42:05Z,4.0,...,False,True,False,143.110.190.60,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['cloud'], 'cloudP...",2025-09-22 11:38:07,"CMS, VA",2,NaN
5,Success,170.80.76.38,6755399460008358,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-22T11:16:45Z,4.0,...,False,True,False,170.80.76.38,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['eol-product'], '...",2025-09-22 11:38:06,"HHS, IHS",2,NaN
6,Success,172.232.209.254,6755399460008058,2025-07-02T12:05:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-20T18:22:31Z,4.0,...,False,True,False,172.232.209.254,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'hostNames': ['172-232-209...",2025-09-22 11:38:07,"HHS, VA",2,NaN
7,Success,190.107.232.146,6755399460008389,2025-07-02T12:05:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-20T18:22:57Z,4.0,...,False,True,False,190.107.232.146,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['vpn'], 'country'...",2025-09-22 11:38:07,"DHA, VA",2,NaN
8,Success,104.131.6.219,5265076,2025-01-23T19:51:12Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-22T10:54:35Z,3.0,...,False,True,False,104.131.6.219,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'tags': ['cloud'], 'cloudP...",2025-09-22 11:38:06,"HRSA, IHS, NIH",3,NaN
9,Success,185.220.101.30,6755399459075899,2025-06-24T16:51:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-21T12:52:49Z,4.0,...,False,True,False,185.220.101.30,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'Shodan', 'hostNames': ['berlin01.to...",2025-09-22 11:38:08,"HHS, OS",2,NaN


In [16]:
otx_df

,search_term,reputation,geo,whois,open_ports,link,timestamp,observed_by,partner_count,base_id,base_indicator,base_type,base_title,base_description,base_content,base_access_type,base_access_reason
0,134.199.159.23,0,{},http://whois.domaintools.com/134.199.159.23,[],https://otx.alienvault.com/indicator/ip/134.19...,2025-09-22 11:38:06,"HRSA, VA",2,4.078883e+09,134.199.159.23,IPv4,,,,public,
1,222.59.173.105,0,{},http://whois.domaintools.com/222.59.173.105,[],https://otx.alienvault.com/indicator/ip/222.59...,2025-09-22 11:38:08,"HHS, NIH",2,4.047330e+09,222.59.173.105,IPv4,,,,public,
2,74.91.112.216,0,{},http://whois.domaintools.com/74.91.112.216,[],https://otx.alienvault.com/indicator/ip/74.91....,2025-09-22 11:38:06,"DHA, HRSA",2,2.559741e+08,74.91.112.216,IPv4,,,,public,
3,104.21.20.66,0,{},http://whois.domaintools.com/104.21.20.66,[],https://otx.alienvault.com/indicator/ip/104.21...,2025-09-22 11:38:06,"DHA, HRSA",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,143.110.190.60,0,{},http://whois.domaintools.com/143.110.190.60,[],https://otx.alienvault.com/indicator/ip/143.11...,2025-09-22 11:38:07,"CMS, VA",2,4.053470e+09,143.110.190.60,IPv4,,,,public,
5,170.80.76.38,0,{},http://whois.domaintools.com/170.80.76.38,[],https://otx.alienvault.com/indicator/ip/170.80...,2025-09-22 11:38:06,"HHS, IHS",2,4.079025e+09,170.80.76.38,IPv4,,,,public,
6,172.232.209.254,0,{},http://whois.domaintools.com/172.232.209.254,[],https://otx.alienvault.com/indicator/ip/172.23...,2025-09-22 11:38:07,"HHS, VA",2,4.049592e+09,172.232.209.254,IPv4,,,,public,
7,190.107.232.146,0,{},http://whois.domaintools.com/190.107.232.146,[],https://otx.alienvault.com/indicator/ip/190.10...,2025-09-22 11:38:07,"DHA, VA",2,4.079025e+09,190.107.232.146,IPv4,,,,public,
8,104.131.6.219,0,{},http://whois.domaintools.com/104.131.6.219,[],https://otx.alienvault.com/indicator/ip/104.13...,2025-09-22 11:38:06,"HRSA, IHS, NIH",3,3.905724e+09,104.131.6.219,IPv4,,,,public,
9,185.220.101.30,0,{},http://whois.domaintools.com/185.220.101.30,[],https://otx.alienvault.com/indicator/ip/185.22...,2025-09-22 11:38:08,"HHS, OS",2,2.122849e+08,185.220.101.30,IPv4,,,,public,


In [10]:
import os
from datetime import datetime

import pandas as pd
import docx
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# ── Paths ─────────────────────────────────────────────────────────────────────
TEMPLATE_PATH = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx"
OUTPUT_DIR   = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def _to_date_str(val):
    """Return YYYY-MM-DD or 'N/A'."""
    if pd.isna(val) or val == "N/A":
        return "N/A"
    try:
        ts = pd.to_datetime(val, errors="coerce", utc=True)
        if pd.isna(ts):
            return "N/A"
        return ts.date().isoformat()
    except Exception:
        return "N/A"

def _add_hyperlink(paragraph, url, text=None):
    """Add a clickable hyperlink to a paragraph (python-docx low-level)."""
    text = text or url
    part = paragraph.part
    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)

    hyperlink = OxmlElement('w:hyperlink')
    hyperlink.set(qn('r:id'), r_id)

    new_run = OxmlElement('w:r')
    rPr = OxmlElement('w:rPr')
    rStyle = OxmlElement('w:rStyle')
    rStyle.set(qn('w:val'), 'Hyperlink')
    rPr.append(rStyle)
    new_run.append(rPr)

    t = OxmlElement('w:t')
    t.text = text
    new_run.append(t)
    hyperlink.append(new_run)

    paragraph._p.append(hyperlink)

def _replace_placeholder_with_text(paragraph, placeholder, replacement):
    """
    Replace {{placeholder}} inside a paragraph without nuking run formatting.
    If not found in this paragraph, do nothing.
    """
    if placeholder not in paragraph.text:
        return False

    # rebuild runs in place
    new_text = []
    for run in paragraph.runs:
        new_text.append(run.text)
    full = "".join(new_text)

    full = full.replace(placeholder, replacement)

    # wipe and write back as one run (simple & safe)
    for run in paragraph.runs:
        run.text = ""
    if paragraph.runs:
        paragraph.runs[0].text = full
    else:
        paragraph.add_run(full)
    return True

def _replace_placeholder_with_hyperlinks(paragraph, placeholder, links):
    """
    Replace {{placeholder}} with a list of hyperlinks (one per line).
    """
    if placeholder not in paragraph.text:
        return False

    # Clear the paragraph and rebuild
    for r in paragraph.runs:
        r.text = ""
    paragraph.text = paragraph.text.replace(placeholder, "")

    if not links:
        paragraph.add_run("N/A")
        return True

    for i, link in enumerate(links):
        _add_hyperlink(paragraph, link)
        if i < len(links) - 1:
            paragraph.add_run().add_break()
    return True

def consolidate_sources(vt_df, otx_df):
    """ Consolidate links from both DataFrames per search_term. """
    def _links_from(df, colname):
        if df is None or df.empty:
            return pd.DataFrame(columns=["search_term", colname])
        out = df.groupby("search_term")["link"].apply(lambda x: ", ".join(x.dropna().astype(str))).reset_index()
        out.columns = ["search_term", colname]
        return out

    vt_links  = _links_from(vt_df, "vt_links")
    otx_links = _links_from(otx_df, "otx_links")

    if vt_links.empty and otx_links.empty:
        return pd.DataFrame(columns=["search_term", "sources"])

    consolidated = pd.merge(vt_links, otx_links, on="search_term", how="outer")
    consolidated["vt_links"]  = consolidated["vt_links"].fillna("")
    consolidated["otx_links"] = consolidated["otx_links"].fillna("")
    consolidated["sources"] = consolidated[["vt_links", "otx_links"]].apply(
        lambda x: ", ".join([p for p in x if p]).strip(", ").strip(), axis=1
    )
    return consolidated[["search_term", "sources"]]

def populate_table(table, data):
    """ Populate the table rows from data. """
    for _, row in data.iterrows():
        cells = table.add_row().cells
        cells[0].text = str(row.get('search_term', 'N/A'))
        cells[1].text = str(row.get('type', 'N/A'))
        cells[2].text = _to_date_str(row.get('observed_date', 'N/A'))

        # observed_by: prefer observed_by_otx else observed_by
        observed_list = []
        if 'observed_by_otx' in row and pd.notna(row['observed_by_otx']):
            observed_list.extend([v.strip() for v in str(row['observed_by_otx']).split(',') if v.strip()])
        elif 'observed_by' in row and pd.notna(row['observed_by']):
            observed_list.extend([v.strip() for v in str(row['observed_by']).split(',') if v.strip()])
        cells[3].text = "\n".join(sorted(set(observed_list))) if observed_list else "N/A"

        cells[4].text = str(row.get('notes', ''))

def find_indicator_table(doc: Document):
    """Find the table whose first row contains 'Indicators/Identifiers'."""
    for tbl in doc.tables:
        try:
            header_text = " | ".join(c.text for c in tbl.rows[0].cells)
        except IndexError:
            continue
        if "Indicators/Identifiers" in header_text:
            return tbl
    return None

def fill_word_template(template_path, output_dir, df, partners_lookup):
    if not os.path.exists(template_path):
        print(f"Template not found: {template_path}")
        return
    if df is None or df.empty:
        print("No data for report.")
        return

    try:
        doc = Document(template_path)

        # Table population (unchanged)
        table = find_indicator_table(doc)
        if table is not None:
            populate_table(table, df)

        # ── Compute context for placeholders ──────────────────────────────────
        # multiple indicators possible
        indicators = list(pd.Series(df['search_term'].dropna().astype(str)).unique()) if 'search_term' in df.columns else ['Unnamed_Indicator']
        search_term_first = indicators[0]
        indicator_count   = len(indicators)

        # prefer showing the first indicator + count if multiple
        ip_placeholder_text = search_term_first if indicator_count == 1 else f"{search_term_first} (+{indicator_count - 1} more)"

        asn_value   = str(df['asn'].dropna().astype(str).iloc[0])   if 'asn'   in df.columns and df['asn'].notna().any()   else 'N/A'
        whois_value = str(df['whois'].dropna().astype(str).iloc[0]) if 'whois' in df.columns and df['whois'].notna().any() else 'N/A'

        # choose a single weblink if available
        weblink_value = ""
        for col in ("webLink", "link"):
            if col in df.columns and df[col].notna().any():
                weblink_value = df[col].dropna().astype(str).iloc[0]
                if weblink_value:
                    break

        # partners: if multiple indicators, prefer first match in partners_lookup
        partners_value = "N/A"
        for st in indicators:
            if st in partners_lookup and partners_lookup[st]:
                partners_value = partners_lookup[st]
                break

        # gather all unique sources across the group
        sources = []
        if "sources" in df.columns:
            for srcs in df["sources"].dropna().astype(str).unique():
                for src in [s.strip() for s in srcs.split(",") if s.strip()]:
                    sources.append(src)
        sources = list(dict.fromkeys(sources))  # dedupe, order-preserving

        # ── Replace placeholders ──────────────────────────────────────────────
        for para in doc.paragraphs:
            _replace_placeholder_with_text(para, "{{ipaddress}}", ip_placeholder_text)
            _replace_placeholder_with_text(para, "{{asn}}",       asn_value)
            _replace_placeholder_with_text(para, "{{whois}}",     whois_value)
            _replace_placeholder_with_text(para, "{{partners}}",  partners_value)

            if "{{weblink}}" in para.text:
                for r in para.runs:
                    r.text = ""
                para.text = para.text.replace("{{weblink}}", "")
                if weblink_value:
                    _add_hyperlink(para, weblink_value)
                else:
                    para.add_run("N/A")

            _replace_placeholder_with_hyperlinks(para, "{{sources}}", sources)

        # ── Group-aware filename ──────────────────────────────────────────────
        # Use group_id if present; else first indicator
        group_part = None
        if 'group_id' in df.columns and df['group_id'].notna().any():
            group_part = str(df['group_id'].dropna().iloc[0])

        base_name = f"Group_{group_part}" if group_part else search_term_first
        base_name = base_name.replace(":", "_").replace("/", "_")

        current_date = datetime.now().strftime("%Y-%m-%d")
        folder_path  = os.path.join(output_dir, current_date)
        os.makedirs(folder_path, exist_ok=True)

        out_path = os.path.join(folder_path, f"I&W_Report_{base_name}.docx")
        doc.save(out_path)
        print(f"Saved report: {out_path}")

    except Exception as e:
        ident = (df['search_term'].iloc[0] if 'search_term' in df.columns and not df.empty else 'Unknown')
        print(f"Error while generating report for {ident}: {e}")

# ── Orchestration ─────────────────────────────────────────────────────────────
def main(vt_df, otx_df, recent_tags):
    # Normalize VT cols we need
    if vt_df is not None and not vt_df.empty:
        vt_df = vt_df.rename(columns={'ip': 'search_term', 'webLink': 'link'})

    # Normalize recent_tags to search_term, keep only relevant cols
    if isinstance(recent_tags, pd.DataFrame) and not recent_tags.empty:
        recent_norm = recent_tags.rename(columns={'summary': 'search_term'}).copy()
        keep_cols = [c for c in ['search_term', 'observations', 'type', 'partners', 'observed_date', 'webLink', 'group_id'] if c in recent_norm.columns]
        recent_norm = recent_norm[keep_cols]
        partners_lookup = (
            recent_tags.set_index('summary')['partners'].astype(str).to_dict()
            if 'summary' in recent_tags.columns and 'partners' in recent_tags.columns
            else {}
        )
    else:
        recent_norm = pd.DataFrame(columns=['search_term'])
        partners_lookup = {}

    # Combine VT/OTX on search_term
    if vt_df is None or vt_df.empty:
        combined_df = otx_df.copy() if otx_df is not None else pd.DataFrame(columns=['search_term'])
    else:
        combined_df = pd.merge(vt_df, otx_df, on='search_term', how='outer', suffixes=('_vt', '_otx'))

    # Consolidate sources
    sources_df = consolidate_sources(vt_df, otx_df)
    combined_df = pd.merge(combined_df, sources_df, on='search_term', how='left')

    # Attach recent_norm (brings in group_id if present)
    combined_df = pd.merge(combined_df, recent_norm, on='search_term', how='left')

    # Nothing to do?
    if 'search_term' not in combined_df.columns or combined_df['search_term'].dropna().empty:
        print("No indicators to generate.")
        return combined_df

    # ── NEW: Group-by group_id if present ────────────────────────────────────
    has_group = 'group_id' in combined_df.columns and combined_df['group_id'].notna().any()

    if has_group:
        # Reports for each non-null group_id
        for gid, gdf in combined_df.dropna(subset=['group_id']).groupby('group_id', dropna=True):
            fill_word_template(
                template_path=TEMPLATE_PATH,
                output_dir=OUTPUT_DIR,
                df=gdf.sort_values(['search_term', 'observed_date'], na_position='last'),
                partners_lookup=partners_lookup,
            )
        # Also handle any rows without a group_id (single-indicator reports)
        orphan_df = combined_df[combined_df['group_id'].isna()]
        for indicator in orphan_df['search_term'].dropna().unique():
            indicator_df = orphan_df[orphan_df['search_term'] == indicator]
            fill_word_template(TEMPLATE_PATH, OUTPUT_DIR, indicator_df, partners_lookup)
    else:
        # Fallback: one report per indicator (existing behavior)
        for indicator in combined_df['search_term'].dropna().unique():
            indicator_df = combined_df[combined_df['search_term'] == indicator]
            fill_word_template(TEMPLATE_PATH, OUTPUT_DIR, indicator_df, partners_lookup)

    return combined_df


# ── Script entry (expects vt_df / otx_df / recent_tags to be defined upstream) ─
if __name__ == "__main__":
    try:
        # If running in a notebook/script that already defined vt_df, otx_df, recent_tags
        # fall back to None if not present.
        vt_df_local = globals().get("vt_df", None)
        otx_df_local = globals().get("otx_df", None)
        recent_tags_local = globals().get("recent_tags", pd.DataFrame())

        main(vt_df_local, otx_df_local, recent_tags_local)
    except Exception as e:
        print(f"Run failed: {e}")


Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-09-22\I&W_Report_Group_5629499543134345.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-09-22\I&W_Report_Group_5629499544001057.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-09-22\I&W_Report_104.131.6.219.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-09-22\I&W_Report_104.21.20.66.docx
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports\2025-09-22\I&W_Report_138.197.101.95.docx
